# 03. Model Training
Training LSTM and other variants based on experiment mode.

In [ ]:
# Model Training Experiment Configuration
import os
import re
import json
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams.update({'font.size': 18})
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.rcParams['legend.fontsize'] = 16
import torch

# --- EXPERIMENT SETTINGS ---
# Modes: 'FEATURE_COMPARISON', 'MODEL_COMPARISON', 'INSTRUMENT_COMPARISON'
EXPERIMENT_MODE = 'FEATURE_COMPARISON'

# Defaults (used when not being compared)
FIXED_MODEL = 'lstm'
FIXED_FEATURE = 'mfcc'
FIXED_INSTRUMENT = 'raw_audio'

# --- GLOBAL PATHS ---
BASE_PATH = Path('./input')
PROCESSED_PATH = Path('./processed')
OUTPUT_PATH = Path('./outputs')

# --- DYNAMIC LISTS FOR FILTERING ---
if EXPERIMENT_MODE == 'FEATURE_COMPARISON':
    MODELS_TO_TEST = [FIXED_MODEL]
    FEATURES_TO_TEST = ['mfcc', 'chromagram', 'mfcc_chroma']
    INSTRUMENTS_TO_TEST = [FIXED_INSTRUMENT]
elif EXPERIMENT_MODE == 'MODEL_COMPARISON':
    MODELS_TO_TEST = ['rnn', 'gru', 'lstm', 'bilstm', 'transformer']
    FEATURES_TO_TEST = [FIXED_FEATURE]
    INSTRUMENTS_TO_TEST = [FIXED_INSTRUMENT]
elif EXPERIMENT_MODE == 'INSTRUMENT_COMPARISON':
    MODELS_TO_TEST = [FIXED_MODEL]
    FEATURES_TO_TEST = [FIXED_FEATURE]
    INSTRUMENTS_TO_TEST = [
        'bass', 'guitar', 'guitar_piano', 'guitar_piano_bass', 
        'no_vocals', 'piano', 'raw_audio', 'vocals'
    ]

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f'Experiment Mode: {EXPERIMENT_MODE}')


In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import KFold
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# --- Dataset & Models ---

class ChordDataset(Dataset):
    def __init__(self, sequences):
        self.X = [torch.from_numpy(s['X']) for s in sequences]
        self.y = [torch.from_numpy(s['y']) for s in sequences]
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

def collate_fn(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([x.size(0) for x in xs])
    x_pad = pad_sequence(xs, batch_first=True, padding_value=0)
    y_pad = pad_sequence(ys, batch_first=True, padding_value=-100)
    return x_pad, y_pad, lengths

class RNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x, lengths=None):
        out, _ = self.rnn(x)
        return self.fc(out)

class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x, lengths=None):
        out, _ = self.gru(x)
        return self.fc(out)

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x, lengths=None):
        out, _ = self.lstm(x)
        return self.fc(out)

class BiLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
    def forward(self, x, lengths=None):
        out, _ = self.lstm(x)
        return self.fc(out)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, nhead=4, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=nhead, batch_first=True, dropout=dropout)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x, lengths=None):
        x = self.input_proj(x)
        out = self.transformer(x)
        return self.fc(out)

def get_model(model_key, input_dim, output_dim):
    if model_key == 'rnn': return RNNModel(input_dim, 128, output_dim)
    if model_key == 'gru': return GRUModel(input_dim, 128, output_dim)
    if model_key == 'lstm': return LSTMModel(input_dim, 128, output_dim)
    if model_key == 'bilstm': return BiLSTMModel(input_dim, 128, output_dim)
    if model_key == 'transformer': return TransformerModel(input_dim, 128, output_dim)
    raise ValueError(f'Unknown model: {model_key}')


In [ ]:
def train_model(model, train_loader, val_loader, epochs=50):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    history = {'train_loss': [], 'val_acc': []}
    
    best_acc = 0
    for epoch in range(epochs):
        model.train()
        t_loss = 0
        for xb, yb, lens in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits.view(-1, logits.size(-1)), yb.view(-1))
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb, lens in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                preds = logits.argmax(dim=-1)
                mask = yb != -100
                correct += (preds[mask] == yb[mask]).sum().item()
                total += mask.sum().item()
        
        acc = correct / total if total > 0 else 0
        history['train_loss'].append(t_loss/len(train_loader))
        history['val_acc'].append(acc)
        if acc > best_acc: best_acc = acc
        if (epoch+1) % 10 == 0: print(f'Epoch {epoch+1}: Loss {t_loss/len(train_loader):.4f}, Acc {acc:.4f}')
    
    return history, best_acc

# --- MAIN TRAINING LOOP ---
processed_file = PROCESSED_PATH / 'master_processed_data.pkl'
if not processed_file.exists():
    raise FileNotFoundError(f'{processed_file} not found. Run preprocessing notebook first.')
    
with open(processed_file, 'rb') as f:
    data_dict = pickle.load(f)

results = []
for inst in INSTRUMENTS_TO_TEST:
    for feat in FEATURES_TO_TEST:
        for model_key in MODELS_TO_TEST:
            key = f'{inst}_{feat}'
            if key not in data_dict: 
                print(f'Skipping {key} (not in processed data)')
                continue
            
            print(f'\n>>> Training {model_key.upper()} on {inst.upper()} with {feat.upper()} features...')
            entry = data_dict[key]
            seqs = entry['sequences']
            
            # 5-Fold Cross Validation
            kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
            fold_accs = []
            for fold, (train_idx, val_idx) in enumerate(kf.split(seqs)):
                print(f'  --- Fold {fold+1}/5 ---')
                train_seqs = [seqs[i] for i in train_idx]
                val_seqs = [seqs[i] for i in val_idx]
                train_ds = ChordDataset(train_seqs)
                val_ds = ChordDataset(val_seqs)
                
                train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
                val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)
                
                model = get_model(model_key, len(entry['feat_cols']), len(entry['classes'])).to(DEVICE)
                hist, best_acc = train_model(model, train_loader, val_loader, epochs=50)
                fold_accs.append(best_acc)
            
            mean_acc = np.mean(fold_accs)
            std_acc = np.std(fold_accs)
            print(f'  --> CV Mean Acc: {mean_acc:.4f} ± {std_acc:.4f}')
            
            results.append({
                'instrument': inst, 'feature': feat, 'model': model_key, 
                'best_acc': mean_acc, 'std_acc': std_acc, 'fold_accs': fold_accs
            })

with open(OUTPUT_PATH / f'results_{EXPERIMENT_MODE.lower()}.json', 'w') as f:
    json.dump(results, f)
print('\nAll training sessions complete.')